# 13 - External Evaluation Scaled L78

Roda uma avaliação externa maior com `lm-evaluation-harness`, comparando `student_pruned_base` contra `l78_adapter`.

Objetivo desta etapa: sair do `--limit 100` e testar `--limit 500` nas mesmas tarefas principais, sem ainda tentar provar runtime packed ternário.

Este notebook avalia qualidade funcional via HF/PEFT:
- Base: `checkpoints/student_pruned`
- L78: `checkpoints/student_pruned` + adapter `checkpoints/student_ternary_recovered_l78`

Ele não executa `deploy/mlp_ternary_2bit.bin`.


In [ ]:
# Auto-bootstrap para Colab/local. Rode esta célula primeiro.
from pathlib import Path
import os, sys, json, time, subprocess, shlex, zipfile

IN_COLAB = 'google.colab' in sys.modules
PROJECT_NAME = 'gemma4_theory'
COLAB_DRIVE_ROOT = '/content/drive/MyDrive/RPT'
LOCAL_ROOT = str(Path.cwd().resolve())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path(COLAB_DRIVE_ROOT) / PROJECT_NAME
else:
    PROJECT_ROOT = Path(LOCAL_ROOT) / PROJECT_NAME if Path(LOCAL_ROOT).name != PROJECT_NAME else Path(LOCAL_ROOT)

DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
CHECKPOINTS_DIR = PROJECT_ROOT / 'checkpoints'
DEPLOY_DIR = PROJECT_ROOT / 'deploy'
OUT_DIR = RESULTS_DIR / 'external_eval_scaled'
for p in [DATA_DIR, RESULTS_DIR, CHECKPOINTS_DIR, DEPLOY_DIR, OUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('CHECKPOINTS_DIR =', CHECKPOINTS_DIR)
print('OUT_DIR =', OUT_DIR)


In [ ]:
# Instala dependências necessárias para lm_eval.
# Em Colab T4, esta instalação costuma ser suficiente para HF/PEFT + lm-evaluation-harness.
import subprocess, sys
subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    '-U',
    'lm-eval',
    'accelerate',
    'sentencepiece',
    'peft',
    'transformers',
    'safetensors',
    'datasets',
])


In [ ]:
# Configuração da avaliação escalada.
BASE_MODEL = CHECKPOINTS_DIR / 'student_pruned'
L78_ADAPTER = CHECKPOINTS_DIR / 'student_ternary_recovered_l78'

if not BASE_MODEL.exists():
    raise FileNotFoundError(f'Base model não encontrado: {BASE_MODEL}')
if not L78_ADAPTER.exists():
    raise FileNotFoundError(f'Adapter l78 não encontrado: {L78_ADAPTER}')

MODEL_ARGS = {
    'student_pruned_base': f'pretrained={BASE_MODEL},trust_remote_code=True',
    'l78_adapter': f'pretrained={BASE_MODEL},peft={L78_ADAPTER},trust_remote_code=True',
}

TASKS_SCALED = ['arc_easy', 'arc_challenge', 'hellaswag', 'truthfulqa_mc1']
BATCH_SIZE = 1
LIMIT = 500
EVAL_LABEL = f'scaled{LIMIT}'
RESUME_EXISTING = True

commands = []
for model_label, model_args in MODEL_ARGS.items():
    task_str = ','.join(TASKS_SCALED)
    out_file = OUT_DIR / f'{model_label}_{EVAL_LABEL}.json'
    cmd = (
        f'lm_eval --model hf '
        f'--model_args "{model_args}" '
        f'--tasks {task_str} '
        f'--batch_size {BATCH_SIZE} '
        f'--limit {LIMIT} '
        f'--output_path {out_file}'
    )
    commands.append({
        'model_label': model_label,
        'eval_label': EVAL_LABEL,
        'tasks': TASKS_SCALED,
        'limit': LIMIT,
        'command': cmd,
        'output_path': str(out_file),
    })

print(json.dumps(commands, ensure_ascii=False, indent=2))


In [ ]:
# Salva plano da avaliação antes de rodar.
plan = {
    'models': {
        'student_pruned_base': str(BASE_MODEL),
        'l78_adapter': str(L78_ADAPTER),
    },
    'model_args': MODEL_ARGS,
    'tasks': TASKS_SCALED,
    'limit': LIMIT,
    'batch_size': BATCH_SIZE,
    'commands': commands,
    'interpretation': {
        'purpose': 'Scale external validation from limit=100 to limit=500.',
        'baseline': 'student_pruned_base',
        'candidate': 'l78_adapter',
        'warning': 'This validates HF/PEFT functional quality, not packed ternary runtime speed.',
        'pass_signal': 'L78 remains near student_pruned_base across the scaled external tasks.',
    },
}
plan_path = OUT_DIR / f'external_eval_scaled_plan_{EVAL_LABEL}.json'
plan_path.write_text(json.dumps(plan, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('saved', plan_path)


In [ ]:
# Helpers para lidar com arquivos timestampados criados pelo lm_eval.
def latest_matching_result(output_path):
    path = Path(output_path)
    candidates = []
    if path.exists():
        candidates.append(path)
    candidates.extend(path.parent.glob(f'{path.stem}_*.json'))
    skip = {
        f'external_eval_scaled_plan_{EVAL_LABEL}.json',
        f'external_eval_scaled_summary_{EVAL_LABEL}.json',
        f'external_eval_scaled_comparison_{EVAL_LABEL}.json',
        f'external_eval_scaled_run_log_{EVAL_LABEL}.json',
    }
    candidates = [p for p in candidates if p.name not in skip]
    return max(candidates, key=lambda p: p.stat().st_mtime) if candidates else None


def read_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def compact_lm_eval_result(path):
    data = read_json(path)
    compact = {}
    for task, metrics in data.get('results', {}).items():
        compact[task] = {
            k.replace(',none', ''): v
            for k, v in metrics.items()
            if k in ('acc,none', 'acc_norm,none')
        }
    return {
        'path': str(path),
        'metrics': compact,
        'n_samples': data.get('n-samples'),
        'config': data.get('config'),
    }


In [ ]:
# Executa as avaliações escaladas em sequência.
# Se RESUME_EXISTING=True, pula qualquer saída já existente para evitar gastar GPU de novo.
run_log = []
for item in commands:
    existing = latest_matching_result(item['output_path'])
    if RESUME_EXISTING and existing is not None:
        print(f"Pulando {item['model_label']} porque já existe: {existing}")
        run_log.append({**item, 'status': 'skipped_existing', 'result_path': str(existing)})
        continue

    print('\n' + '=' * 100)
    print('Executando:', item['model_label'], item['eval_label'])
    print(item['command'])
    start = time.time()
    completed = subprocess.run(item['command'], shell=True)
    elapsed = round(time.time() - start, 2)
    result_path = latest_matching_result(item['output_path'])
    run_log.append({
        **item,
        'status': 'ok' if completed.returncode == 0 else 'failed',
        'returncode': completed.returncode,
        'elapsed_seconds': elapsed,
        'result_path': str(result_path) if result_path else None,
    })
    if completed.returncode != 0:
        raise RuntimeError(f"lm_eval falhou para {item['model_label']} com returncode={completed.returncode}")

run_log_path = OUT_DIR / f'external_eval_scaled_run_log_{EVAL_LABEL}.json'
run_log_path.write_text(json.dumps(run_log, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('\nRun log salvo em:', run_log_path)
print(json.dumps(run_log, ensure_ascii=False, indent=2))


In [ ]:
# Consolida resultados e compara L78 contra base.
summary = {}
for item in commands:
    result_path = latest_matching_result(item['output_path'])
    key = f"{item['model_label']}_{item['eval_label']}"
    if result_path is None:
        summary[key] = {'exists': False, 'requested_path': item['output_path']}
    else:
        summary[key] = {'exists': True, **compact_lm_eval_result(result_path)}

base_key = f'student_pruned_base_{EVAL_LABEL}'
l78_key = f'l78_adapter_{EVAL_LABEL}'
base = summary.get(base_key, {})
l78 = summary.get(l78_key, {})

def metric_value(entry, task, metric):
    if not entry.get('exists'):
        return None
    return entry.get('metrics', {}).get(task, {}).get(metric)

comparison = {
    'note': f'lm_eval com --limit {LIMIT}; métrica mais robusta que limit=100, mas ainda não substitui benchmark completo sem limite.',
    'baseline_key': base_key,
    'candidate_key': l78_key,
    'rows': [],
}
all_tasks = sorted(set(base.get('metrics', {})) | set(l78.get('metrics', {})))
for task in all_tasks:
    metrics = sorted(set(base.get('metrics', {}).get(task, {})) | set(l78.get('metrics', {}).get(task, {})))
    for metric in metrics:
        b = metric_value(base, task, metric)
        v = metric_value(l78, task, metric)
        comparison['rows'].append({
            'task': task,
            'metric': metric,
            'student_pruned_base': b,
            'l78_adapter': v,
            'delta_l78_minus_base': None if b is None or v is None else round(v - b, 6),
        })

summary_path = OUT_DIR / f'external_eval_scaled_summary_{EVAL_LABEL}.json'
comparison_path = OUT_DIR / f'external_eval_scaled_comparison_{EVAL_LABEL}.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
comparison_path.write_text(json.dumps(comparison, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

print('Summary salvo em:', summary_path)
print('Comparison salvo em:', comparison_path)
print(json.dumps(comparison, ensure_ascii=False, indent=2))


In [ ]:
# Exporta artefatos da etapa 13 para download.
artifact_paths = [
    OUT_DIR / f'external_eval_scaled_plan_{EVAL_LABEL}.json',
    OUT_DIR / f'external_eval_scaled_run_log_{EVAL_LABEL}.json',
    OUT_DIR / f'external_eval_scaled_summary_{EVAL_LABEL}.json',
    OUT_DIR / f'external_eval_scaled_comparison_{EVAL_LABEL}.json',
]
for item in commands:
    result_path = latest_matching_result(item['output_path'])
    if result_path is not None:
        artifact_paths.append(result_path)

zip_path = OUT_DIR / f'external_eval_scaled_{EVAL_LABEL}_artifacts.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in artifact_paths:
        if p.exists():
            zf.write(p, arcname=p.name)

print('ZIP salvo em:', zip_path)
print('Arquivos incluídos:')
for p in artifact_paths:
    print('-', p, 'OK' if p.exists() else 'MISSING')

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
